# Soil Sample Vertical Weighting
**Yuxi Zhang** · Oct 18, 2026 · [@yuxi-research](https://github.com/yuxi-research)

📘 This notebok


⚙️ **Main Processing Steps**

1. **Process**

   * Convert weight percentage of Hydrogen, SOC, and/or Lattice water in soil samples into equivalent water.
   * Read weather data (pressure, temperature) from rover data.
   * Calculate farm average in each individual depth.
   * Iterate to estimate effective depth (D86) and vertical weighting. Methods: 'conv' or 'schron2017'.
   * Apply the vertical weighting to the profiles of soil moisture and water-equivalent at each survey and site.
   * Save the weighted soil dataframe.
  
💬 **Notes**

* Two vertical weighting methods are provided, both according to the description on schron2017. Options: 'conv' and 'schron2017'.  
* Schron 2017 revised method accounts for the impact of pressure, temperature and soil mositure on the effective penetration depth. Conventional method only accounts for soil moisture (Schrön 2017, Eq 3).  

* The ierative process of Schrön (2017) only run steps 1, 2, 3, 5. Step 4 was skipped because hirizontal weighting is seperately applied in B2PK process. Here we used the horizontally uniform average to estimate vertical weighting.

📥 **Input**

* `./data_lv2/{farm_name}/data/`

  * `{farm_name}_{date}__rover_clean.shp`: Cleaned rover data.
* `settings.py`: Configuration file specifying the farm name, target CRS, folder structure, and file paths.

📤 **Output**

* `./data_lv2/{farm_name}/data/`

  * `{farm_name}_soilsamples_weighted.shp`: Rover data matched with neutron monitoring flux data.

🧩 **Parameters**

* `weight_method`: Options: 'conv' and 'schron2017'. Any other keywards uses the 'schron2017' method.  

Note: 

Reference:
Schrön, M., Köhli, M., Scheiffele, L., Iwema, J., Bogena, H. R., Lv, L.,…Zacharias, S. (2017). Improving calibration and validation of cosmic-ray neutron sensors in the light of spatial sensitivity. Hydrol. Earth Syst. Sci., 21(10), 5009-5030. https://doi.org/10.5194/hess-21-5009-2017 


### Settings

In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import re
import xarray as xr


In [2]:
# Path settings
from global_settings import *
farm_name = os.environ.get("FARM_NAME", "Amaroo")
dirs = ProjectPaths(farm_name=farm_name)
proj_crs = dirs.proj_crs

In [3]:
weight_method = 'schron2017' # conv or schron2017. Both according to the description on schron2017.

In [4]:
# Function to calculate SOC equivalent water and lattice water
def calculate_water_equivalent(ds):
    """
    Calculate water-equivalent carbon content and lattice water from a dataset or dataframe.
    All input units are in weight percentage (%). The output will be in volumetric water content (m3/m3) if bulk density (BD) is provided, otherwise in gravimetric water content (g/g).

    If 'Hydrogen' (H from CNH method) is present, it assumes it releases hydrogen from SOM and lattice water. 
    Else, it uses the 'Carbon' (C) content to estimate SOC water-equivalent, and uses 'Clay' content to estimate lattice water.
    
    Parameters:
    ds (xr.Dataset or pd.DataFrame): Input dataset or dataframe containing the necessary variables.
    Returns:
    xr.Dataset or pd.DataFrame: Output dataset or dataframe with calculated water-equivalent carbon content and lattice water.

    """
    if type(ds) == xr.Dataset:
        ds_vars = list(ds.data_vars)
        ds_out = xr.Dataset()
    else:
        ds_vars = ds.columns.tolist()
        ds_out = pd.DataFrame(index=ds.index)
    
    
    # Water-equivalent carbon content estimation
    if 'H' in ds_vars:
        # CNH analysis method. Assuming it releases hydrogen from SOM and lattice water. 
        # The factor 18/2 is the stoichiometric ratio of water to hydrogen in water.
        watHydrogen = ds['H'] * 18 / 2 / 100
    else:
        watHydrogen = np.zeros_like(ds[ds_vars[0]]) * np.nan
    if 'C' in ds_vars:
        # 1.25 factor is the stoichiometric ratio of hydrogen equivalent-water to carbon in organic matter, assuming organic matter consists of cellulose (C6H10O5)n.
        watCarbon = ds['C'] * 1.25 / 100
    else:
        watCarbon = np.zeros_like(ds[ds_vars[0]]) * np.nan

    # Lattice water
    if 'Clay' in ds_vars:
        watLattice = 0.00075 * ds['Clay'] + 0.0121
    else:
        watLattice = np.zeros_like(ds[ds_vars[0]]) * np.nan

    # Volumetric soil moisture estimation
    if 'BD' in ds_vars:
        ds_out['watHydrogenVol'] = watHydrogen * ds['BD']
        ds_out['watLatticeVol'] = watLattice * ds['BD']
        ds_out['watCarbonVol'] = watCarbon * ds['BD']

    else:
        ds_out['watHydrogenGrav'] = watHydrogen
        ds_out['watLatticeGrav'] = watLattice
        ds_out['watCarbonGrav'] = watCarbon

    return ds_out

### Read cleaned soil sample data

In [5]:
# Read the soil moisture data all depths
pth_soil = os.path.join(dirs.lv1_soil, f'{farm_name}_soilsamples_fillna.csv')
df_soil = pd.read_csv(pth_soil)
df_soil = df_soil.drop_duplicates()


### Calculate hydrogen pool from other sources (SOC and lattice water)

In [6]:
# Calculate SOC equivalent water and lattice water
df_soil = pd.concat([df_soil, calculate_water_equivalent(df_soil)], axis=1)

### Read processed rover data

In [7]:
# Find all rover data and dates
regex = re.compile(rf'^{farm_name}_(.*)_rover_dataframe.csv$')
filesnames_rover = [f for f in os.listdir(dirs.lv2_data) if regex.match(f)]
filesnames_rover.sort()
print(filesnames_rover)
dates_str = [regex.match(f).group(1) for f in filesnames_rover]


In [8]:
df_rover = []
for filename_rover in filesnames_rover:
    # Read rover data
    df_rover.append(pd.read_csv(os.path.join(dirs.lv2_data, filename_rover)))

df_rover = pd.concat(df_rover, ignore_index=True)
mean_values = df_rover[['PTB110_mb', 'RH_CS215', 'T_CS215', 'SurveyDate']].groupby('SurveyDate').mean().reset_index()


### Define vertical weighting

In [9]:
# Calculate the farm average soil moisture and other properties for each depth interval.
df_soil_hmean = df_soil.groupby(['Date', 'DepthUpper', 'DepthLower']).mean(numeric_only=True).reset_index()

df_soil_hmean['DepthMid'] = (df_soil['DepthUpper'] + df_soil['DepthLower']) / 2
df_soil_hmean['Thickness'] = df_soil['DepthLower'] - df_soil['DepthUpper']

# Determine the target soil moisture column name based on available columns in the dataframe.
if 'SMVol' in df_soil_hmean.columns:
    sm_name = 'SMVol'
else:
    sm_name = 'SMGrav'

## Schron 2017 method or conventional method


In [10]:
from utils.crns_vertical_weighting import *

In [11]:
df_vweights_norm_all = []
for date_str in dates_str:
    print(f'Processing date: {date_str}')
    # Get the profile of the soil moisture for the specific date, average of the soil moisture for each depth.
    df_soil_hmean_day = df_soil_hmean[df_soil_hmean['Date'] == date_str].copy()
    df_soil_hmean_day.sort_values(by=['DepthUpper', 'DepthLower'], inplace=True)

    # Mean values for the day
    mean_values_day = mean_values[mean_values['SurveyDate'] == date_str].copy()
    if 'BD' in df_soil_hmean_day.columns:
        mean_bd = df_soil_hmean_day['BD'].mean()
    else:
        mean_bd = 1  # or some default value
    mean_pressure = mean_values_day['PTB110_mb'].values[0]
    mean_h_veg = 0  # Assuming no vegetation for the calculation

    # Initial vertical weights
    vweights = np.ones((len(df_soil_hmean_day)))
    vweights_norm = vweights / sum(vweights)
    init_theta = -1
    new_theta = np.dot(df_soil_hmean_day[sm_name], vweights_norm)

    # break
    # Loop
    iteration = 0
    while np.abs(new_theta - init_theta) > 0.0001:
        # Prevent infinite loop
        iteration += 1
        if iteration > 20:
            print('Warning: Maximum iterations reached')
            break

        # Loop until convergence
        init_theta = new_theta
        if weight_method == 'conv':
            # Use the convolution method
            d86 = calc_D86_conv(init_theta)
            vweights = df_soil_hmean_day['DepthMid'].apply(calc_wd_conv, args=(d86,))
            vweights_norm = vweights / np.sum(vweights)
        else:
            # Else use the Schron2017 method
            # Parameter r=1. COSMOS Guide: It is recommend to set r =1 as a proxy for the average measurement depth in the footprint.
            r_rescaled = calc_r_rescaled(r=1, p=mean_pressure, theta=init_theta, H_veg=mean_h_veg)
            d86 = calc_d86(r_rescaled=r_rescaled, theta=init_theta, rho_bulk=mean_bd)
            vweights = calc_wd(d=df_soil_hmean_day['DepthMid'], d86=d86)
            vweights_norm = vweights / np.sum(vweights)

        new_theta = np.dot(df_soil_hmean_day[sm_name], vweights_norm)

        print(f'Iteration: {iteration}, init_theta: {init_theta:.4f}, new_theta: {new_theta:.4f}, d86: {d86:.4f}, vweights_norm: {vweights_norm.values}')

    df_soil_hmean_day['vertical_weight'] = vweights_norm
    df_vweights_norm = df_soil_hmean_day.set_index(['Date', 'DepthUpper', 'DepthLower'])[['vertical_weight']].reset_index()
    df_vweights_norm_all.append(df_vweights_norm)

### Merge the estimated weights into the original soil dataframe

In [12]:
# Merge the estimated weights into the original soil dataframe
df_vweights_norm_all = pd.concat(df_vweights_norm_all, ignore_index=True)
df_vweights_norm_all
df_soil = df_soil.merge(df_vweights_norm_all, on=['Date', 'DepthUpper', 'DepthLower'], how='left', suffixes=('', '_norm'))


In [13]:
# Select variables in columns to apply weighting
cols = [c for c in df_soil.columns if c not in ['Date', 'SiteID', 'Longitude', 'Latitude', 'DepthUpper', 'DepthLower', 'vertical_weight']]
cols = [c for c in cols if df_soil.dtypes[c] in [np.float64, np.float32, np.int64, np.int32]]


In [14]:
# Apply the vertical weights to the soil properties for each date and site
df_soil_weighted = []
for col in cols:
    df_col = df_soil.groupby(['Date', 'SiteID', 'Longitude', 'Latitude']
                             )[[col, 'vertical_weight']].apply(lambda x: np.dot(x[col], x['vertical_weight'])
                                           ).to_frame(name=col)
    df_soil_weighted.append(df_col)

df_soil_weighted = pd.concat(df_soil_weighted, axis=1)
df_soil_weighted = df_soil_weighted.round(4).reset_index()

# Export the weighted soil properties to a CSV file
df_soil_weighted.to_csv(os.path.join(dirs.lv2_data, f'{farm_name}_soilsamples_weighted.csv'), index=False)

In [15]:
df_soil_weighted